In [1]:
# 0

import warnings
from pathlib import Path

import pandas as pd
import numpy as np

from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [2]:
# 1

XLSX = 'kospi_data.xlsx'

EXCLUDE_FIN = ['A000680', 'A000880', 'A005110', 'A008060', 'A015020', 'A023590',
               'A026890', 'A030190', 'A034310', 'A210980', 'A244920']

def build_gd(path=XLSX, out='gd.parquet'):
    if Path(out).exists():
        df = pd.read_parquet(out)
        print(f'{out} loaded. {len(df)}행  {df["qtr"].min()} ~ {df["qtr"].max()}')
        return df
    raw = pd.read_excel(path, sheet_name='34_gd', header=None)
    body = raw.iloc[14:, :2].dropna()
    df = pd.DataFrame({
        'qtr': pd.PeriodIndex(body.iloc[:, 0].astype(str), freq='Q'),
        'gd': pd.to_numeric(body.iloc[:, 1], errors='coerce'),
    }).dropna().sort_values('qtr').reset_index(drop=True)
    df.to_parquet(out)
    print(f'{out} written. {len(df)}행  {df["qtr"].min()} ~ {df["qtr"].max()}')
    return df

financials = pd.read_parquet('financials.parquet')
financials = financials[~financials['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)

prices = pd.read_parquet('prices.parquet')
prices = prices[~prices['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)

delisted = pd.read_parquet('delisted.parquet')
delisted = delisted[~delisted['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)

gd = build_gd()

print(f'financials  {financials.shape}  종목 {financials["code"].nunique()}')
print(f'prices      {prices.shape}  종목 {prices["code"].nunique()}')
print(f'delisted    {len(delisted)}개')
print(f'columns: {financials.columns.tolist()}')

gd.parquet loaded. 105행  2000Q1 ~ 2026Q1
financials  (116235, 29)  종목 1107
prices      (353133, 5)  종목 1107
delisted    355개
columns: ['code', 'date', 'actq', 'rectq', 'invtq', 'ppentq', 'atq', 'lctq', 'dlttq', 'dlcq', 'ltq', 'seqq', 'cogsq', 'xsgaq', 'saleq', 'apq', 'opinc', 'xintq', 'ibq', 'txditcq', 'pstkq', 'cheq', 'oancfq', 'dpq', 'dvq', 'cstkq', 'req', 'epsq', 'txtq']


In [3]:
# 2

FORCE_IMPAIR = ['A001600', 'A006210']
FORCE_REORG = ['A019930']
BAD_DEFAULT = ['A011840', 'A018590', 'A001980', 'A010730', 'A005600',
               'A007050', 'A011800', 'A002130']
BAD_SUSPEND = ['A000200', 'A003810', 'A017120', 'A000310', 'A004510',
               'A004550', 'A004740', 'A012600', 'A093230']
FORMAL = ['A003980', 'A012720', 'A009790', 'A012270', 'A008750', 'A003020',
          'A009760', 'A006250', 'A001190', 'A016160', 'A015110', 'A002530',
          'A017320', 'A014080', 'A008500']
MERGER_EXTRA = ['A103160', 'A049770', 'A008000', 'A002250', 'A027390', 'A144620',
                'A015350', 'A138250', 'A068400', 'A001880', 'A003410', 'A115390',
                'A034300', 'A010420', 'A450140', 'A019440']
NAN_CODES = ['A950010', 'A011160', 'A102280']

BAD_CATS = ['감사의견', '자본잠식', '정리절차', '부도', '실질심사']
NONBAD_CATS = ['합병', '자진', '형식요건']

d = delisted.copy()
d['cat'] = '미분류'
d.loc[d['reason_cat'] == '감사의견', 'cat'] = '감사의견'
d.loc[d['reason_cat'] == '자본잠식', 'cat'] = '자본잠식'
d.loc[d['reason_cat'] == '정리절차', 'cat'] = '정리절차'
d.loc[d['reason_cat'] == '합병·해산', 'cat'] = '합병'
d.loc[d['reason_cat'] == '자진·신청', 'cat'] = '자진'
d.loc[d['code'].isin(FORCE_IMPAIR), 'cat'] = '자본잠식'
d.loc[d['code'].isin(FORCE_REORG), 'cat'] = '정리절차'
d.loc[d['code'].isin(BAD_DEFAULT), 'cat'] = '부도'
d.loc[d['code'].isin(BAD_SUSPEND), 'cat'] = '실질심사'
d.loc[d['code'].isin(FORMAL), 'cat'] = '형식요건'
d.loc[d['code'].isin(MERGER_EXTRA), 'cat'] = '합병'
d.loc[d['code'].isin(NAN_CODES), 'cat'] = '미분류'

d['is_bad'] = d['cat'].isin(BAD_CATS).astype(int)
d['excl'] = (d['cat'] == '미분류').astype(int)
d.to_parquet('delisted_cat.parquet')

fin_has = financials.groupby('code')['atq'].apply(lambda s: s.notna().any())
fin_codes = set(fin_has[fin_has].index)
h = d[d['code'].isin(fin_codes)]

cnt = pd.DataFrame({
    '전체': d['cat'].value_counts(),
    '재무존재': h['cat'].value_counts(),
}).reindex(BAD_CATS + NONBAD_CATS + ['미분류']).fillna(0).astype(int)
cnt['구분'] = np.where(cnt.index.isin(BAD_CATS), '부실',
                     np.where(cnt.index.isin(NONBAD_CATS), '비부실', '미분류'))

blocks = [cnt[cnt['구분'] == g].sort_values('재무존재', ascending=False)
          for g in ['부실', '비부실', '미분류']]
tbl = pd.concat(blocks).rename_axis('사유').reset_index()[['구분', '사유', '전체', '재무존재']]
display(tbl)

sub = tbl.groupby('구분', sort=False)[['전체', '재무존재']].sum().reindex(['부실', '비부실', '미분류'])
sub.loc['합계'] = sub.sum()
display(sub)

hb = h[h['is_bad'] == 1].copy()
hb['year'] = hb['del_date'].dt.year.astype(int)
yr = (hb.groupby(['year', 'cat']).size().unstack('cat', fill_value=0)
      .reindex(columns=blocks[0].index.tolist(), fill_value=0))
yr['합계'] = yr.sum(axis=1)
yr.loc['합계'] = yr.sum()
display(yr)

,구분,사유,전체,재무존재
0,부실,감사의견,71,70
1,부실,자본잠식,31,23
2,부실,정리절차,31,10
3,부실,실질심사,9,9
4,부실,부도,8,8
5,비부실,합병,130,83
6,비부실,형식요건,15,15
7,비부실,자진,23,14
8,미분류,미분류,37,3


,전체,재무존재
구분,,
부실,150,120
비부실,168,112
미분류,37,3
합계,355,235


cat,감사의견,자본잠식,정리절차,실질심사,부도,합계
year,,,,,,
2000,0,0,1,0,0,1
2001,0,0,4,4,1,9
2002,15,4,1,0,0,20
2003,2,1,1,0,1,5
2004,5,1,1,1,3,11
2005,5,2,2,0,1,10
2006,1,0,0,0,0,1
2007,1,4,0,0,0,5
2008,1,0,0,0,0,1


In [4]:
# 3

MIN_MONTHS = 12
FILL_LIMIT = 2
YR0, YR1 = 2002, 2025
VARS = ['NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA']

mkt = pd.read_parquet('market.parquet')
mkt['ym'] = pd.PeriodIndex(mkt['date'], freq='M')
mkt_m = (mkt.assign(g=1 + mkt['kret'] / 100)
         .groupby('ym')['g'].prod().sub(1).rename('mret').reset_index())
mkt_m['year'] = mkt_m['ym'].dt.year

px = prices[['code', 'date', 'ret', 'mktcap']].copy()
px['ym'] = pd.PeriodIndex(px['date'], freq='M')
px['year'] = px['ym'].dt.year
px['mret_i'] = px['ret'] / 100
live = px[px['ret'].notna() | px['mktcap'].notna()].copy()

r = px[px['mret_i'].notna()].copy()
r['g'] = 1 + r['mret_i']
fy = r.groupby(['code', 'year']).agg(g=('g', 'prod'), n=('mret_i', 'count'))
fy = fy[fy['n'] >= MIN_MONTHS]
fy['r_ann'] = fy['g'] - 1
m_ann = (mkt_m.assign(g=1 + mkt_m['mret'])
         .groupby('year')['g'].prod().sub(1).rename('m_ann'))
exret = fy[['r_ann']].join(m_ann, on='year')
exret['EXRET'] = exret['r_ann'] - exret['m_ann']
exret = exret[['EXRET']].reset_index()

reg = (r[['code', 'year', 'ym', 'mret_i']]
       .merge(mkt_m[['ym', 'mret']], on='ym', how='inner'))

def _sigma(g):
    if len(g) < MIN_MONTHS:
        return np.nan
    x, y = g['mret'].values, g['mret_i'].values
    b = np.polyfit(x, y, 1)
    return (y - (b[0] * x + b[1])).std(ddof=2)

sigma = reg.groupby(['code', 'year']).apply(_sigma).rename('SIGMA').reset_index()
sigma.loc[sigma['SIGMA'] <= 0, 'SIGMA'] = np.nan

dec = px[(px['ym'].dt.month == 12) & px['mktcap'].notna()][['code', 'year', 'mktcap']]
tot = dec.groupby('year')['mktcap'].sum().rename('mktcap_tot')
rsize = dec.join(tot, on='year')
rsize['RSIZE'] = np.log(rsize['mktcap'] / rsize['mktcap_tot'])
rsize = rsize[['code', 'year', 'RSIZE']]

fin = financials.sort_values(['code', 'date']).reset_index(drop=True).copy()
pq = pd.PeriodIndex(fin['date'], freq='Q')
fin['qtr'] = pq
fin['qidx'] = pq.year * 4 + (pq.quarter - 1)
g = fin.groupby('code', sort=False)
ok4 = (fin['qidx'] - g['qidx'].shift(3)) == 3
fin['ibq_ttm'] = np.where(
    ok4, g['ibq'].apply(lambda x: x.rolling(4, min_periods=4).sum())
    .reset_index(level=0, drop=True), np.nan)
acc = fin[fin['qtr'].dt.quarter == 2][['code', 'qtr', 'ibq_ttm', 'ltq', 'atq']].copy()
acc['year'] = acc['qtr'].dt.year
den = acc['atq'].where(acc['atq'] > 0)
acc['NITA'] = acc['ibq_ttm'] / den
acc['TLTA'] = acc['ltq'] / den
acc = acc[['code', 'year', 'NITA', 'TLTA']]

X = (rsize.merge(exret, on=['code', 'year'], how='outer')
     .merge(sigma, on=['code', 'year'], how='outer')
     .merge(acc, on=['code', 'year'], how='outer'))

full = (X[['code']].drop_duplicates()
        .merge(pd.DataFrame({'year': range(X['year'].min(), X['year'].max() + 1)}), how='cross')
        .merge(X, on=['code', 'year'], how='left')
        .sort_values(['code', 'year']))

gg = full.groupby('code', sort=False)
for c in VARS:
    filled = gg[c].ffill(limit=FILL_LIMIT)
    full[f'{c}_f'] = filled
    full[f'{c}_fillage'] = np.where(
        full[c].notna(), 0,
        np.where(filled.notna(),
                 full['year'] - gg['year'].transform(
                     lambda s: s).where(full[c].notna()).groupby(full['code']).ffill(),
                 np.nan))

full['obs_year'] = full['year'] + 1
X2 = full.drop(columns='year')

dc_all = pd.read_parquet('delisted_cat.parquet')
excl_codes = set(dc_all.query('excl == 1')['code'])
dc = dc_all[dc_all['excl'] == 0][['code', 'del_date', 'is_bad']].copy()
dc['del_year'] = dc['del_date'].dt.year.astype(int)

alive = live[['code', 'year']].drop_duplicates()
alive = alive[(alive['year'] >= YR0) & (alive['year'] <= YR1)]
alive = alive[~alive['code'].isin(excl_codes)]
alive = alive.merge(dc[['code', 'del_year', 'is_bad']], on='code', how='left')
alive = alive[alive['del_year'].isna() | (alive['year'] <= alive['del_year'])]
alive['y'] = ((alive['del_year'] == alive['year']) & (alive['is_bad'] == 1)).astype(int)
alive['censored'] = ((alive['del_year'] == alive['year']) & (alive['is_bad'] == 0)).astype(int)

panel = (alive.rename(columns={'year': 'obs_year'})[['code', 'obs_year', 'y', 'censored']]
         .merge(X2, on=['code', 'obs_year'], how='left'))

FVARS = [f'{c}_f' for c in VARS]
AGES = [f'{c}_fillage' for c in VARS]
panel['n_filled'] = (panel[AGES] > 0).sum(axis=1)
panel['any_filled'] = (panel['n_filled'] > 0).astype(int)

comp_f = panel[FVARS].notna().all(axis=1)
lo = panel.loc[comp_f, FVARS].quantile(0.01)
hi = panel.loc[comp_f, FVARS].quantile(0.99)
for c in VARS:
    panel[f'{c}_w'] = panel[f'{c}_f'].clip(lo[f'{c}_f'], hi[f'{c}_f'])

panel = panel[['code', 'obs_year', 'y', 'censored', 'n_filled', 'any_filled']
              + VARS + FVARS + AGES + [f'{c}_w' for c in VARS]]
panel.to_parquet('shumway_panel.parquet')

raw_c = panel[VARS].notna().all(axis=1)

ov = pd.DataFrame(
    {'값': [len(panel), panel['code'].nunique(),
           f'{panel["obs_year"].min()}~{panel["obs_year"].max()}',
           int(panel['y'].sum()), int(panel['censored'].sum())]},
    index=pd.Index(['패널 행', '종목', '기간', '사건', '절단'], name='항목'))
display(ov)

samp = pd.DataFrame({
    '행': [len(panel), int(raw_c.sum()), int(comp_f.sum()), int(panel['any_filled'].sum())],
    '사건': [int(panel['y'].sum()), int(panel.loc[raw_c, 'y'].sum()),
            int(panel.loc[comp_f, 'y'].sum()), int(panel.loc[panel['any_filled'] == 1, 'y'].sum())],
}, index=pd.Index(['전체 패널', '대체 전 완비', '대체 후 완비', '대체 관측'], name='표본'))
samp['사건률%'] = (samp['사건'] / samp['행'] * 100).round(2)
display(samp)

byyr = (panel[comp_f].groupby('obs_year')
        .agg(표본=('y', 'size'), 사건=('y', 'sum')))
byyr['사건률%'] = (byyr['사건'] / byyr['표본'] * 100).round(2)
byyr.loc['합계'] = [byyr['표본'].sum(), byyr['사건'].sum(),
                  round(byyr['사건'].sum() / byyr['표본'].sum() * 100, 2)]
byyr.index.name = '연도'
display(byyr)

,값
항목,
패널 행,16657
종목,962
기간,2002~2025
사건,102
절단,80


,행,사건,사건률%
표본,,,
전체 패널,16657,102,0.61
대체 전 완비,15831,90,0.57
대체 후 완비,15870,100,0.63
대체 관측,46,11,23.91


,표본,사건,사건률%
연도,,,
2002,576.0,19.0,3.30
2003,600.0,5.0,0.83
2004,606.0,10.0,1.65
2005,601.0,9.0,1.50
2006,597.0,1.0,0.17
2007,609.0,5.0,0.82
2008,617.0,1.0,0.16
2009,632.0,12.0,1.90
2010,634.0,11.0,1.74


In [5]:
# 4

import statsmodels.api as sm

p = pd.read_parquet('shumway_panel.parquet')
VARS = ['NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA']
WVARS = [f'{c}_w' for c in VARS]

MODELS = {
    'M1 시장': ['RSIZE', 'EXRET', 'SIGMA'],
    'M2 회계': ['NITA', 'TLTA'],
    'M3 결합': ['NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA'],
    'M4 SIGMA제외': ['NITA', 'TLTA', 'RSIZE', 'EXRET'],
}

def fit_hazard(df, cols, suffix='_w'):
    use = [f'{c}{suffix}' for c in cols]
    d = df.dropna(subset=use + ['y'])
    X = sm.add_constant(d[use].astype(float))
    X.columns = ['Intercept'] + cols
    res = sm.Logit(d['y'].astype(float), X).fit(disp=0)
    n_firm = d['code'].nunique()
    n_obs = len(d)
    adj = n_obs / n_firm
    out = pd.DataFrame({
        '계수': res.params.round(3),
        'chi2': (res.tvalues ** 2).round(2),
        'chi2_보정': ((res.tvalues ** 2) / adj).round(2),
    })
    out['p_보정'] = stats.chi2.sf(out['chi2_보정'], 1).round(3)
    meta = dict(firms=n_firm, obs=n_obs, events=int(d['y'].sum()),
                adj=adj, llf=res.llf, prsq=res.prsquared)
    return out, meta, res

coefs, metas = {}, {}
for name, cols in MODELS.items():
    out, meta, _ = fit_hazard(p, cols)
    coefs[name] = out.rename_axis('변수')
    metas[name] = meta

fit = pd.DataFrame(metas).T[['firms', 'obs', 'events', 'adj', 'prsq']]
fit.columns = ['기업', 'firm-year', '사건', '기업당연수', 'pseudo-R2']
fit[['기업', 'firm-year', '사건']] = fit[['기업', 'firm-year', '사건']].astype(int)
fit['기업당연수'] = fit['기업당연수'].round(2)
fit['pseudo-R2'] = fit['pseudo-R2'].round(3)
fit.index.name = '모형'
print('=== 모형별 적합도 ===')
display(fit)

cf = pd.concat(coefs, names=['모형'])
print('=== 모형별 계수 (chi2_보정 = chi2 / 기업당연수) ===')
display(cf)

ORDER = ['Intercept'] + VARS
FOOT = ['사건', '기업당연수']

wide = cf['계수'].unstack('모형').reindex(ORDER)[list(MODELS)]
wide = pd.concat([wide, fit[FOOT].T[list(MODELS)]])
wide.index.name = '변수'

pwide = cf['p_보정'].unstack('모형').reindex(ORDER)[list(MODELS)]
pwide = pd.concat([pwide, fit[FOOT].T[list(MODELS)]])
pwide.index.name = '변수'

print('=== 모형 × 변수 계수 비교 ===')
display(wide.round(3))
print('=== 모형 × 변수 p_보정 비교 ===')
display(pwide.round(3))

ref_a = pd.DataFrame({
    '계수': [-12.027, -0.503, -2.072, 9.834],
    'chi2': [39.27, 8.06, 11.14, 11.03],
    'p': [.000, .005, .001, .001],
}, index=['Intercept', 'RSIZE', 'EXRET', 'SIGMA'])
ref_b = pd.DataFrame({
    '계수': [-13.303, -1.982, 3.593, -0.467, -1.809, 5.791],
    'chi2': [30.79, .88, 6.90, 5.24, 6.52, 2.47],
    'p': [.000, .348, .009, .022, .011, .116],
}, index=['Intercept', 'NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA'])

ref = pd.concat({'Panel A 시장': ref_a, 'Panel B 결합': ref_b},
                names=['패널', '변수'])
ref_n = pd.DataFrame({
    '기업': [2894, 2497], 'firm-year': [33621, 28664], '사건': [291, 239],
    '기업당연수': [11.62, 11.48],
}, index=pd.Index(['Panel A 시장', 'Panel B 결합'], name='패널'))
print('=== 원문 Table 6 ===')
display(ref_n)
display(ref)

=== 모형별 적합도 ===


,기업,firm-year,사건,기업당연수,pseudo-R2
모형,,,,,
M1 시장,939,15970,101,17.01,0.271
M2 회계,939,15952,101,16.99,0.247
M3 결합,936,15870,100,16.96,0.337
M4 SIGMA제외,936,15873,101,16.96,0.305


=== 모형별 계수 (chi2_보정 = chi2 / 기업당연수) ===


계수    chi2  chi2_보정   p_보정
모형         변수                                       
M1 시장      Intercept -14.833  151.37     8.90  0.003
           RSIZE      -0.857   48.89     2.87  0.090
           EXRET      -1.813   44.81     2.63  0.105
           SIGMA       8.510  112.07     6.59  0.010
M2 회계      Intercept  -7.759  401.28    23.62  0.000
           NITA       -6.249  128.67     7.57  0.006
           TLTA        3.810   44.57     2.62  0.106
M3 결합      Intercept -13.855  140.35     8.28  0.004
           NITA       -2.949   21.77     1.28  0.258
           TLTA        3.063   31.17     1.84  0.175
           RSIZE      -0.601   26.36     1.55  0.213
           EXRET      -1.180   19.61     1.16  0.281
           SIGMA       5.655   35.26     2.08  0.149
M4 SIGMA제외 Intercept -13.536  136.38     8.04  0.005
           NITA       -4.207   47.23     2.78  0.095
           TLTA        3.425   38.62     2.28  0.131
           RSIZE      -0.635   29.86     1.76  0.185
           EXRET      -1.108   13.66     0.81  0.368

=== 모형 × 변수 계수 비교 ===


...

=== 모형 × 변수 p_보정 비교 ===


모형,M1 시장,M2 회계,M3 결합,M4 SIGMA제외
변수,,,,
Intercept,0.003,0.000,0.004,0.005
NITA,NaN,0.006,0.258,0.095
TLTA,NaN,0.106,0.175,0.131
RSIZE,0.090,NaN,0.213,0.185
EXRET,0.105,NaN,0.281,0.368
SIGMA,0.010,NaN,0.149,NaN
사건,101.000,101.000,100.000,101.000
기업당연수,17.010,16.990,16.960,16.960


=== 원문 Table 6 ===


,기업,firm-year,사건,기업당연수
패널,,,,
Panel A 시장,2894,33621,291,11.62
Panel B 결합,2497,28664,239,11.48


계수   chi2      p
패널         변수                             
Panel A 시장 Intercept -12.027  39.27  0.000
           RSIZE      -0.503   8.06  0.005
           EXRET      -2.072  11.14  0.001
           SIGMA       9.834  11.03  0.001
Panel B 결합 Intercept -13.303  30.79  0.000
           NITA       -1.982   0.88  0.348
           TLTA        3.593   6.90  0.009
           RSIZE      -0.467   5.24  0.022
           EXRET      -1.809   6.52  0.011
           SIGMA       5.791   2.47  0.116

In [6]:
# 5

import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

p = pd.read_parquet('shumway_panel.parquet')
VARS = ['NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA']
FVARS = [f'{c}_f' for c in VARS]

MODELS = {
    'M1 시장': ['RSIZE', 'EXRET', 'SIGMA'],
    'M2 회계': ['NITA', 'TLTA'],
    'M3 결합': VARS,
    'M4 SIGMA제외': ['NITA', 'TLTA', 'RSIZE', 'EXRET'],
}

TRAIN_MIN = 2002
TEST_START = 2008
TEST_END = 2025

def walk_forward(cols):
    use = [f'{c}_f' for c in cols]
    rows = []
    for t in range(TEST_START, TEST_END + 1):
        tr = p[(p['obs_year'] >= TRAIN_MIN) & (p['obs_year'] < t)].dropna(subset=use + ['y'])
        te = p[p['obs_year'] == t].dropna(subset=use + ['y']).copy()
        if tr['y'].sum() < 5 or len(te) == 0:
            continue
        lo = tr[use].quantile(0.01)
        hi = tr[use].quantile(0.99)
        Xtr = sm.add_constant(tr[use].clip(lo, hi, axis=1).astype(float))
        Xte = sm.add_constant(te[use].clip(lo, hi, axis=1).astype(float), has_constant='add')
        try:
            res = sm.Logit(tr['y'].astype(float), Xtr).fit(disp=0)
        except Exception:
            continue
        te['ph'] = res.predict(Xte)
        te['dec'] = te.groupby('obs_year')['ph'].rank(pct=True, ascending=False)
        te['dec'] = np.ceil(te['dec'] * 10).clip(1, 10).astype(int)
        rows.append(te[['code', 'obs_year', 'y', 'ph', 'dec']])
    return pd.concat(rows, ignore_index=True) if rows else None

res_all = {}
for name, cols in MODELS.items():
    res_all[name] = walk_forward(cols)

auc = pd.DataFrame(
    [{'관측': len(r), '사건': int(r['y'].sum()),
      'AUC': round(roc_auc_score(r['y'], r['ph']), 3)}
     for r in res_all.values()],
    index=pd.Index(list(res_all), name='모형'))
print('=== walk-forward 표본 외 (예측 2008~2025) ===')
display(auc)

tab = {}
for name, r in res_all.items():
    ev = r[r['y'] == 1]
    v = ev['dec'].value_counts(normalize=True).mul(100).reindex(range(1, 11)).fillna(0)
    tab[name] = v.round(1)
tab = pd.DataFrame(tab)
tab.loc['6-10'] = tab.loc[6:10].sum().round(1)
tab = tab.drop(index=range(6, 11))
tab.index = pd.Index(['1분위', '2분위', '3분위', '4분위', '5분위', '6-10분위'], name='분위')
print('=== 십분위 포착률 (%, 사건 중 각 분위 비중) ===')
display(tab)

ref = pd.DataFrame({
    '시장전용': [69.0, 10.6, 7.8, 5.0, 2.8, 4.8],
    '회계+시장': [75.0, 12.5, 6.3, 1.8, 0.9, 3.5],
}, index=pd.Index(['1분위', '2분위', '3분위', '4분위', '5분위', '6-10분위'], name='분위'))
print('=== 원문 Table 7 (1962-83 추정, 1984-92 예측. 시장전용 142건, 결합 112건) ===')
display(ref)

r3 = res_all['M3 결합']
yr3 = r3.groupby('obs_year')['y'].agg(['size', 'sum'])
yr3.columns = ['관측', '사건']
yr3.index.name = '연도'
print('=== 연도별 검증 사건 수 (M3) ===')
display(yr3)

pd.concat([r.assign(model=k) for k, r in res_all.items()],
          ignore_index=True).to_parquet('shumway_oos.parquet')

=== walk-forward 표본 외 (예측 2008~2025) ===


,관측,사건,AUC
모형,,,
M1 시장,12340,52,0.922
M2 회계,12349,52,0.862
M3 결합,12281,51,0.947
M4 SIGMA제외,12284,52,0.921


=== 십분위 포착률 (%, 사건 중 각 분위 비중) ===


,M1 시장,M2 회계,M3 결합,M4 SIGMA제외
분위,,,,
1분위,75.0,65.4,80.4,75.0
2분위,13.5,7.7,11.8,9.6
3분위,5.8,11.5,5.9,9.6
4분위,0.0,0.0,2.0,1.9
5분위,1.9,7.7,0.0,1.9
6-10분위,3.8,7.6,0.0,1.9


=== 원문 Table 7 (1962-83 추정, 1984-92 예측. 시장전용 142건, 결합 112건) ===


,시장전용,회계+시장
분위,,
1분위,69.0,75.0
2분위,10.6,12.5
3분위,7.8,6.3
4분위,5.0,1.8
5분위,2.8,0.9
6-10분위,4.8,3.5


=== 연도별 검증 사건 수 (M3) ===


,관측,사건
연도,,
2008,617,1
2009,632,12
2010,634,11
2011,637,4
2012,652,5
2013,659,2
2014,666,0
2015,667,3
2016,671,2


In [7]:
# 6

import statsmodels.api as sm

p = pd.read_parquet('shumway_panel.parquet')
VARS = ['NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA']

MODELS = {
    'M1 시장': ['RSIZE', 'EXRET', 'SIGMA'],
    'M2 회계': ['NITA', 'TLTA'],
    'M3 결합': ['NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA'],
    'M4 SIGMA제외': ['NITA', 'TLTA', 'RSIZE', 'EXRET'],
}

def three_se(df, cols, suffix='_w'):
    use = [f'{c}{suffix}' for c in cols]
    d = df.dropna(subset=use + ['y']).copy()
    X = sm.add_constant(d[use].astype(float))
    X.columns = ['Intercept'] + cols
    y = d['y'].astype(float)

    plain = sm.Logit(y, X).fit(disp=0)
    clust = sm.Logit(y, X).fit(disp=0, cov_type='cluster',
                               cov_kwds={'groups': d['code']})

    adj = len(d) / d['code'].nunique()
    chi2_plain = plain.tvalues ** 2
    chi2_shum = chi2_plain / adj
    chi2_clust = clust.tvalues ** 2

    return pd.DataFrame({
        '계수': plain.params.round(3),
        'p_보정없음': stats.chi2.sf(chi2_plain, 1).round(3),
        'p_Shumway': stats.chi2.sf(chi2_shum, 1).round(3),
        'p_클러스터': stats.chi2.sf(chi2_clust, 1).round(3),
    }).rename_axis('변수')

se_all = {name: three_se(p, cols) for name, cols in MODELS.items()}

print('=== 표준오차 방식별 p값 ===')
display(pd.concat(se_all, names=['모형']))

print('=== M3 결합: 세 방식 비교 ===')
display(se_all['M3 결합'])

print('=== p값 방식별 유의 변수 수 (5% 기준, 절편 제외) ===')
cnt = pd.DataFrame({
    name: {c: (df.drop(index='Intercept')[c] < 0.05).sum()
           for c in ['p_보정없음', 'p_Shumway', 'p_클러스터']}
    for name, df in se_all.items()
}).T
cnt.index.name = '모형'
display(cnt)

=== 표준오차 방식별 p값 ===


계수  p_보정없음  p_Shumway  p_클러스터
모형         변수                                          
M1 시장      Intercept -14.833     0.0      0.003   0.000
           RSIZE      -0.857     0.0      0.090   0.000
           EXRET      -1.813     0.0      0.105   0.000
           SIGMA       8.510     0.0      0.010   0.000
M2 회계      Intercept  -7.759     0.0      0.000   0.000
           NITA       -6.249     0.0      0.006   0.000
           TLTA        3.810     0.0      0.105   0.000
M3 결합      Intercept -13.855     0.0      0.004   0.000
           NITA       -2.949     0.0      0.257   0.000
           TLTA        3.063     0.0      0.175   0.000
           RSIZE      -0.601     0.0      0.212   0.000
           EXRET      -1.180     0.0      0.282   0.001
           SIGMA       5.655     0.0      0.149   0.000
M4 SIGMA제외 Intercept -13.536     0.0      0.005   0.000
           NITA       -4.207     0.0      0.095   0.000
           TLTA        3.425     0.0      0.131   0.000
           RSIZE      -0.635     0.0      0.185   0.000
           EXRET      -1.108     0.0      0.369   0.012

=== M3 결합: 세 방식 비교 ===


,계수,p_보정없음,p_Shumway,p_클러스터
변수,,,,
Intercept,-13.855,0.0,0.004,0.000
NITA,-2.949,0.0,0.257,0.000
TLTA,3.063,0.0,0.175,0.000
RSIZE,-0.601,0.0,0.212,0.000
EXRET,-1.180,0.0,0.282,0.001
SIGMA,5.655,0.0,0.149,0.000


=== p값 방식별 유의 변수 수 (5% 기준, 절편 제외) ===


,p_보정없음,p_Shumway,p_클러스터
모형,,,
M1 시장,3,1,3
M2 회계,2,1,2
M3 결합,5,0,5
M4 SIGMA제외,4,0,4


In [15]:
# 7

import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

p = pd.read_parquet('shumway_panel.parquet')
VARS = ['NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA']
WVARS = [f'{c}_w' for c in VARS]
HS = [1, 2, 3, 4, 5]

dc = pd.read_parquet('delisted_cat.parquet')
bad = dc[(dc['is_bad'] == 1) & (dc['excl'] == 0)].copy()
bad['del_year'] = bad['del_date'].dt.year.astype(int)
BAD_YR = dict(zip(bad['code'], bad['del_year']))
nonbad = dc[(dc['is_bad'] == 0) & (dc['excl'] == 0)].copy()
nonbad['del_year'] = nonbad['del_date'].dt.year.astype(int)
CENS_YR = dict(zip(nonbad['code'], nonbad['del_year']))
YMAX = int(p['obs_year'].max())

def build_h(h):
    d = p.copy()
    d['bad_yr'] = d['code'].map(BAD_YR)
    d['cens_yr'] = d['code'].map(CENS_YR)
    d['y_h'] = (d['bad_yr'] == d['obs_year'] + h - 1).astype(int)
    keep = d['obs_year'] + h - 1 <= YMAX
    keep &= d['bad_yr'].isna() | (d['obs_year'] <= d['bad_yr'] - h + 1)
    keep &= d['cens_yr'].isna() | (d['obs_year'] <= d['cens_yr'] - h + 1)
    return d[keep].copy()

coefs, p_clu, p_shu, p_raw, fits = {}, {}, {}, {}, {}
for h in HS:
    d = build_h(h).dropna(subset=WVARS + ['y_h'])
    X = sm.add_constant(d[WVARS].astype(float))
    X.columns = ['Intercept'] + VARS
    yy = d['y_h'].astype(float)
    r0 = sm.Logit(yy, X).fit(disp=0)
    rc = sm.Logit(yy, X).fit(disp=0, cov_type='cluster',
                             cov_kwds={'groups': d['code']})
    adj = len(d) / d['code'].nunique()
    k = f'h={h}'
    coefs[k] = r0.params.round(3)
    p_raw[k] = pd.Series(stats.chi2.sf(r0.tvalues ** 2, 1), index=r0.params.index).round(3)
    p_shu[k] = pd.Series(stats.chi2.sf((r0.tvalues ** 2) / adj, 1), index=r0.params.index).round(3)
    p_clu[k] = pd.Series(stats.chi2.sf(rc.tvalues ** 2, 1), index=r0.params.index).round(3)
    fits[k] = dict(사건=int(d['y_h'].sum()), obs=len(d),
                   기업당연수=round(adj, 2), pseudoR2=round(r0.prsquared, 3))

fit_df = pd.DataFrame(fits).T
print('=== 지평별 적합도 ===')
display(fit_df)

print('=== 지평별 계수 ===')
display(pd.DataFrame(coefs))

print('=== 지평별 p (클러스터 강건, 주 추론) ===')
display(pd.DataFrame(p_clu))

print('=== 지평별 p (Shumway 보정) ===')
display(pd.DataFrame(p_shu))

print('=== 지평별 p (보정 없음) ===')
display(pd.DataFrame(p_raw))

def wf(h, cols=VARS):
    use = [f'{c}_f' for c in cols]
    d = build_h(h)
    rows = []
    for t in range(2008, YMAX + 1):
        tr = d[(d['obs_year'] >= 2002) & (d['obs_year'] < t)].dropna(subset=use + ['y_h'])
        te = d[d['obs_year'] == t].dropna(subset=use + ['y_h']).copy()
        if tr['y_h'].sum() < 5 or len(te) == 0:
            continue
        lo, hi = tr[use].quantile(0.01), tr[use].quantile(0.99)
        Xtr = sm.add_constant(tr[use].clip(lo, hi, axis=1).astype(float))
        Xte = sm.add_constant(te[use].clip(lo, hi, axis=1).astype(float), has_constant='add')
        try:
            r = sm.Logit(tr['y_h'].astype(float), Xtr).fit(disp=0)
        except Exception:
            continue
        te['ph'] = r.predict(Xte)
        te['dec'] = np.ceil(te.groupby('obs_year')['ph']
                            .rank(pct=True, ascending=False) * 10).clip(1, 10).astype(int)
        rows.append(te[['code', 'obs_year', 'y_h', 'ph', 'dec']])
    return pd.concat(rows, ignore_index=True) if rows else None

oos = {h: wf(h) for h in HS}
oos_no = {h: wf(h, ['NITA', 'TLTA', 'RSIZE', 'EXRET']) for h in HS}

rows = []
for h in HS:
    r, rn = oos[h], oos_no[h]
    ev = r[r['y_h'] == 1]
    a1 = roc_auc_score(r['y_h'], r['ph'])
    a0 = roc_auc_score(rn['y_h'], rn['ph'])
    rows.append({'h': h, '검증사건': int(r['y_h'].sum()),
                 'AUC': round(a1, 3), 'AUC_SIGMA제외': round(a0, 3),
                 'ΔAUC': round(a1 - a0, 3),
                 '1분위%': round((ev['dec'] == 1).mean() * 100, 1),
                 '1-2분위%': round((ev['dec'] <= 2).mean() * 100, 1),
                 '6-10분위%': round((ev['dec'] >= 6).mean() * 100, 1)})
oos_df = pd.DataFrame(rows).set_index('h')
print('=== 지평별 표본 외 성과 (walk-forward, 2008~2025) ===')
display(oos_df)

tab = {}
for h in HS:
    ev = oos[h][oos[h]['y_h'] == 1]
    v = ev['dec'].value_counts(normalize=True).mul(100).reindex(range(1, 11)).fillna(0)
    tab[f'h={h}'] = v.round(1)
dec_df = pd.DataFrame(tab)
dec_df.loc['6-10'] = dec_df.loc[6:10].sum().round(1)
dec_df = dec_df.drop(index=range(6, 11))
dec_df.index = ['1분위', '2분위', '3분위', '4분위', '5분위', '6-10분위']
print('=== 지평별 십분위 포착률 (%) ===')
display(dec_df)

pd.concat([r.assign(h=h) for h, r in oos.items()],
          ignore_index=True).to_parquet('shumway_oos_h.parquet')

=== 지평별 적합도 ===


,사건,obs,기업당연수,pseudoR2
h=1,100.0,15870.0,16.96,0.337
h=2,84.0,14953.0,16.54,0.219
h=3,78.0,14050.0,15.89,0.189
h=4,66.0,13166.0,15.53,0.152
h=5,54.0,12318.0,15.04,0.110


=== 지평별 계수 ===


,h=1,h=2,h=3,h=4,h=5
Intercept,-13.855,-11.372,-10.562,-9.173,-10.144
NITA,-2.949,-3.900,-2.034,-4.182,-3.239
TLTA,3.063,1.882,1.065,0.565,0.165
RSIZE,-0.601,-0.459,-0.384,-0.292,-0.419
EXRET,-1.180,-0.643,-0.978,-0.365,0.105
SIGMA,5.655,4.405,7.501,5.235,5.134


=== 지평별 p (클러스터 강건, 주 추론) ===


...

,h=1,h=2,h=3,h=4,h=5
Intercept,0.0,0.000,0.000,0.000,0.000
NITA,0.0,0.000,0.006,0.000,0.001
TLTA,0.0,0.001,0.051,0.330,0.798
RSIZE,0.0,0.000,0.000,0.009,0.001
EXRET,0.0,0.008,0.000,0.108,0.624
SIGMA,0.0,0.000,0.000,0.000,0.000


=== 지평별 표본 외 성과 (walk-forward, 2008~2025) ===


,검증사건,AUC,AUC_SIGMA제외,ΔAUC,1분위%,1-2분위%,6-10분위%
h,,,,,,,
1,51,0.947,0.921,0.026,80.4,92.2,0.0
2,51,0.900,0.868,0.033,66.7,80.4,3.9
3,39,0.922,0.862,0.060,71.8,82.1,2.6
4,27,0.839,0.813,0.026,51.9,66.7,14.8
5,22,0.770,0.781,-0.012,40.9,54.5,18.2


=== 지평별 십분위 포착률 (%) ===


,h=1,h=2,h=3,h=4,h=5
1분위,80.4,66.7,71.8,51.9,40.9
2분위,11.8,13.7,10.3,14.8,13.6
3분위,5.9,9.8,12.8,14.8,13.6
4분위,2.0,3.9,2.6,3.7,0.0
5분위,0.0,2.0,0.0,0.0,13.6
6-10분위,0.0,4.0,2.6,14.8,18.1


In [18]:
# 8

import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

oos = pd.read_parquet('shumway_oos_h.parquet')
HS = [1, 2, 3, 4, 5]
CUTS = [0.01, 0.02, 0.05, 0.10, 0.20]

rows = []
for h in HS:
    r = oos[oos['h'] == h]
    n, ev = len(r), int(r['y_h'].sum())
    for c in CUTS:
        k = int(np.ceil(n * c))
        thr = r['ph'].nlargest(k).iloc[-1]
        flag = r['ph'] >= thr
        tp = int((flag & (r['y_h'] == 1)).sum())
        fp = int((flag & (r['y_h'] == 0)).sum())
        fn = ev - tp
        tn = n - tp - fp - fn
        rows.append({
            'h': h, '컷오프': f'{int(c*100)}%', '지정종목': tp + fp,
            '포착(TP)': tp, '오지정(FP)': fp, '누락(FN)': fn,
            '정확률': round(tp / (tp + fp), 4) if tp + fp else np.nan,
            '포착률': round(tp / ev, 3) if ev else np.nan,
            'TypeI(FPR)': round(fp / (fp + tn), 4),
            'TypeII(FNR)': round(fn / ev, 3) if ev else np.nan,
        })
conf = pd.DataFrame(rows).set_index(['h', '컷오프'])
print('=== 컷오프별 판별 성능 ===')
display(conf)

base = oos.groupby('h')['y_h'].agg(['size', 'sum'])
base.columns = ['관측', '사건']
base['기저사건율'] = (base['사건'] / base['관측']).round(4)
print('=== 기저 사건율 ===')
display(base)

lift = conf.reset_index().merge(base['기저사건율'].reset_index(), on='h')
lift['리프트'] = (lift['정확률'] / lift['기저사건율']).round(1)
lift = lift.pivot(index='컷오프', columns='h', values='리프트').reindex(
    [f'{int(c*100)}%' for c in CUTS])
print('=== 리프트 (정확률 / 기저사건율) ===')
display(lift)

RNG = np.random.default_rng(42)
B = 2000

def boot_auc(r):
    codes, inv = np.unique(r['code'].values, return_inverse=True)
    idx_by_firm = [np.where(inv == i)[0] for i in range(len(codes))]
    y = r['y_h'].values
    ph = r['ph'].values
    out = np.empty(B)
    out[:] = np.nan
    for b in range(B):
        pick = RNG.integers(0, len(codes), len(codes))
        idx = np.concatenate([idx_by_firm[i] for i in pick])
        yy = y[idx]
        if yy.min() == yy.max():
            continue
        out[b] = roc_auc_score(yy, ph[idx])
    return out[~np.isnan(out)]

rows = []
for h in HS:
    r = oos[oos['h'] == h]
    b = boot_auc(r)
    rows.append({'h': h, '사건': int(r['y_h'].sum()),
                 'AUC': round(roc_auc_score(r['y_h'], r['ph']), 3),
                 'CI하한': round(np.percentile(b, 2.5), 3),
                 'CI상한': round(np.percentile(b, 97.5), 3),
                 '표준오차': round(b.std(), 3)})
auc_ci = pd.DataFrame(rows).set_index('h')
print('=== AUC 95% 신뢰구간 (기업 클러스터 부트스트랩, B=2000) ===')
display(auc_ci)

=== 컷오프별 판별 성능 ===


지정종목  포착(TP)  오지정(FP)  누락(FN)     정확률    포착률  TypeI(FPR)  TypeII(FNR)
h 컷오프                                                                       
1 1%    123      21      102      30  0.1707  0.412      0.0083        0.588
  2%    246      30      216      21  0.1220  0.588      0.0177        0.412
  5%    615      36      579      15  0.0585  0.706      0.0473        0.294
  10%  1229      42     1187       9  0.0342  0.824      0.0971        0.176
  20%  2457      48     2409       3  0.0195  0.941      0.1970        0.059
2 1%    115      12      103      39  0.1043  0.235      0.0090        0.765
  2%    229      18      211      33  0.0786  0.353      0.0185        0.647
  5%    572      24      548      27  0.0420  0.471      0.0481        0.529
  10%  1144      37     1107      14  0.0323  0.725      0.0972        0.275
  20%  2288      44     2244       7  0.0192  0.863      0.1971        0.137
3 1%    106       8       98      31  0.0755  0.205      0.0093        0.795
  2%    212      15      197      24  0.0708  0.385      0.0187        0.615
  5%    530      23      507      16  0.0434  0.590      0.0481        0.410
  10%  1059      28     1031      11  0.0264  0.718      0.0977        0.282
  20%  2118      35     2083       4  0.0165  0.897      0.1974        0.103
4 1%     98       7       91      20  0.0714  0.259      0.0093        0.741
  2%    196       8      188      19  0.0408  0.296      0.0193        0.704
  5%    489      10      479      17  0.0204  0.370      0.0492        0.630
  10%   977      14      963      13  0.0143  0.519      0.0989        0.481
  20%  1953      17     1936      10  0.0087  0.630      0.1988        0.370
5 1%     90       1       89      21  0.0111  0.045      0.0099        0.955
  2%    180       7      173      15  0.0389  0.318      0.0193        0.682
  5%    449       8      441      14  0.0178  0.364      0.0493        0.636
  10%   898      11      887      11  0.0122  0.500      0.0991        0.500
  20%  1795      12     1783      10  0.0067  0.545      0.1992        0.455

=== 기저 사건율 ===


,관측,사건,기저사건율
h,,,
1,12281,51,0.0042
2,11436,51,0.0045
3,10590,39,0.0037
4,9765,27,0.0028
5,8974,22,0.0025


=== 리프트 (정확률 / 기저사건율) ===


h,1,2,3,4,5
컷오프,,,,,
1%,40.6,23.2,20.4,25.5,4.4
2%,29.0,17.5,19.1,14.6,15.6
5%,13.9,9.3,11.7,7.3,7.1
10%,8.1,7.2,7.1,5.1,4.9
20%,4.6,4.3,4.5,3.1,2.7


=== AUC 95% 신뢰구간 (기업 클러스터 부트스트랩, B=2000) ===


,사건,AUC,CI하한,CI상한,표준오차
h,,,,,
1,51,0.947,0.917,0.972,0.014
2,51,0.900,0.854,0.939,0.022
3,39,0.922,0.888,0.954,0.017
4,27,0.839,0.760,0.907,0.038
5,22,0.770,0.669,0.868,0.052


In [19]:
# 진단

for label, df in [('컷오프별 판별 성능', conf),
                  ('기저 사건율', base),
                  ('리프트', lift),
                  ('AUC 95% 신뢰구간', auc_ci)]:
    print(f'=== {label} ===')
    print(df.to_string())
    print()

=== 컷오프별 판별 성능 ===
       지정종목  포착(TP)  오지정(FP)  누락(FN)     정확률    포착률  TypeI(FPR)  TypeII(FNR)
h 컷오프                                                                       
1 1%    123      21      102      30  0.1707  0.412      0.0083        0.588
  2%    246      30      216      21  0.1220  0.588      0.0177        0.412
  5%    615      36      579      15  0.0585  0.706      0.0473        0.294
  10%  1229      42     1187       9  0.0342  0.824      0.0971        0.176
  20%  2457      48     2409       3  0.0195  0.941      0.1970        0.059
2 1%    115      12      103      39  0.1043  0.235      0.0090        0.765
  2%    229      18      211      33  0.0786  0.353      0.0185        0.647
  5%    572      24      548      27  0.0420  0.471      0.0481        0.529
  10%  1144      37     1107      14  0.0323  0.725      0.0972        0.275
  20%  2288      44     2244       7  0.0192  0.863      0.1971        0.137
3 1%    106       8       98      31  0.0755  0.205      